In [2]:
import numpy as np
import pandas as pd
import fasttext
from src.utils import PROCESSED_DATA_DIR
from src.models.neural_models import train_mlp_fasttext
from src.evaluation import evaluate_model

NEURAL_DATA_DIR = PROCESSED_DATA_DIR / "neural"

X_train_ft_lemma = np.load(NEURAL_DATA_DIR / "X_train_ft.npy")
X_test_ft_lemma = np.load(NEURAL_DATA_DIR / "X_test_ft.npy")

y_train_lemma = np.load(NEURAL_DATA_DIR / "y_train.npy")
y_test_lemma = np.load(NEURAL_DATA_DIR / "y_test.npy")


print("X_train:", X_train_ft_lemma.shape)
print("X_test :", X_test_ft_lemma.shape)
print("y_train:", y_train_lemma.shape)
print("y_test :", y_test_lemma.shape)


# Load the FastText model that was already trained
ft_lemma = fasttext.load_model(
    str(NEURAL_DATA_DIR / "fasttext.bin")
)

mlp_lemma, mlp_pred_lemma, mlp_metrics_lemma = train_mlp_fasttext(
    X_train_ft_lemma,
    X_test_ft_lemma,
    y_train_lemma,
    y_test_lemma,
    ft_lemma,
    evaluate_model
)

mlp_result = pd.DataFrame([{
    "Preprocessing": "Lemmatization",
    "Model": "MLP",
    "Feature": "FastText 300d",
    "Accuracy": mlp_metrics_lemma["Accuracy"],
    "Precision": mlp_metrics_lemma["Precision"],
    "Recall": mlp_metrics_lemma["Recall"],
    "Macro F1": mlp_metrics_lemma["Macro F1"]
}])


print("\n********  MLP RESULT  ********\n")

print(
    mlp_result.to_string(index=False)
)

X_train: (50894, 300)
X_test : (12724, 300)
y_train: (50894,)
y_test : (12724,)

MLP with Self-trained FastText Embeddings
Accuracy: 0.729094624331971
Macro F1: 0.6720168288759836
                 precision    recall  f1-score   support

not_recommended       0.76      0.77      0.77      3222
        no_idea       0.36      0.52      0.42      2107
    recommended       0.89      0.77      0.83      7395

       accuracy                           0.73     12724
      macro avg       0.67      0.69      0.67     12724
   weighted avg       0.77      0.73      0.74     12724


********  MLP RESULT  ********

Preprocessing Model       Feature  Accuracy  Precision   Recall  Macro F1
Lemmatization   MLP FastText 300d  0.729095   0.670212 0.687099  0.672017


In [3]:
from sklearn.model_selection import train_test_split
from src.utils import PROCESSED_DATA_DIR
from src.models.neural_models import (
    prepare_sequences,
    create_class_weights,
    train_cnn
)

# Load already-preprocessed lemma data

df_lemma = pd.read_csv(
    PROCESSED_DATA_DIR / "clean_lemma.csv"
)
df_lemma["text_lemma"] = (
    df_lemma["text_lemma"]
    .fillna("")
    .astype(str)
)

# Recreate the same train/test split
X_train_lemma, X_test_lemma, y_train_lemma, y_test_lemma = train_test_split(
    df_lemma["text_lemma"],
    df_lemma["label"],
    test_size=0.2,
    random_state=42,
    stratify=df_lemma["label"]
)

print("Train size:", len(X_train_lemma))
print("Test size :", len(X_test_lemma))


# Load the FastText model that was already trained
ft_lemma = fasttext.load_model(
    str(NEURAL_DATA_DIR / "fasttext.bin")
)


# build FastText embedding matrix 
(
    tokenizer_lemma,
    X_train_pad_lemma,
    X_test_pad_lemma,
    vocab_size_lemma,
    embedding_dim_lemma,
    embedding_matrix_lemma
) = prepare_sequences(
    X_train_lemma,
    X_test_lemma,
    ft_lemma,
    num_words=20000,
    max_len=100
)

print("X_train_pad:", X_train_pad_lemma.shape)
print("X_test_pad :", X_test_pad_lemma.shape)
print("Vocab size :", vocab_size_lemma)
print("Embedding matrix:", embedding_matrix_lemma.shape)

# Create class weights
class_weights_lemma = create_class_weights(
    y_train_lemma
)

print("Class weights:", class_weights_lemma)

cnn_lemma, history_cnn_lemma, cnn_pred_lemma, cnn_metrics_lemma = train_cnn(
    vocab_size_lemma,
    embedding_dim_lemma,
    embedding_matrix_lemma,
    X_train_pad_lemma,
    X_test_pad_lemma,
    y_train_lemma,
    y_test_lemma,
    class_weights_lemma,
    evaluate_model
)

cnn_result = pd.DataFrame([{
    "Preprocessing": "Lemmatization",
    "Model": "CNN",
    "Feature": "FastText 300d",
    "Accuracy": cnn_metrics_lemma["Accuracy"],
    "Precision": cnn_metrics_lemma["Precision"],
    "Recall": cnn_metrics_lemma["Recall"],
    "Macro F1": cnn_metrics_lemma["Macro F1"]
}])


print("\n********  CNN RESULT  ********\n")

print(
    cnn_result.to_string(index=False)
)

Train size: 50894
Test size : 12724
X_train_pad: (50894, 100)
X_test_pad : (12724, 100)
Vocab size : 20000
Embedding matrix: (20000, 300)
Class weights: {0: np.float64(1.3163149182702256), 1: np.float64(2.012654723771108), 2: np.float64(0.5735763149293933)}


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 100, 300)            │       6,000,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d (Conv1D)                      │ (None, 96, 128)             │         192,128 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ (None, 128)                 │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │             195 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,200,579 (23.65 MB)

 Trainable params: 6,200,579 (23.65 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
716/716 ━━━━━━━━━━━━━━━━━━━━ 63s 70ms/step - accuracy: 0.6952 - loss: 0.7725 - val_accuracy: 0.7183 - val_loss: 0.6780
Epoch 2/20
716/716 ━━━━━━━━━━━━━━━━━━━━ 51s 72ms/step - accuracy: 0.7779 - loss: 0.6287 - val_accuracy: 0.7637 - val_loss: 0.5862
Epoch 3/20
716/716 ━━━━━━━━━━━━━━━━━━━━ 53s 74ms/step - accuracy: 0.8181 - loss: 0.5285 - val_accuracy: 0.7532 - val_loss: 0.6024
Epoch 4/20
716/716 ━━━━━━━━━━━━━━━━━━━━ 55s 76ms/step - accuracy: 0.8670 - loss: 0.4056 - val_accuracy: 0.7674 - val_loss: 0.6419
Epoch 5/20
716/716 ━━━━━━━━━━━━━━━━━━━━ 59s 82ms/step - accuracy: 0.9123 - loss: 0.2792 - val_accuracy: 0.7731 - val_loss: 0.7074
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
398/398 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step

********  CNN RESULTS  **********
                 precision    recall  f1-score   support

not_recommended       0.85      0.73      0.78      3222
        no_idea       0.39      0.58      0.47      2107
    recommended    

In [4]:
from sklearn.model_selection import train_test_split
from src.utils import PROCESSED_DATA_DIR
from src.models.neural_models import (
    prepare_sequences,
    create_class_weights,
    train_rnn
)

# Load already-preprocessed lemma data
df_lemma = pd.read_csv(
    PROCESSED_DATA_DIR / "clean_lemma.csv"
)

df_lemma["text_lemma"] = (
    df_lemma["text_lemma"]
    .fillna("")
    .astype(str)
)


# Recreate the same train/test split
X_train_lemma, X_test_lemma, y_train_lemma, y_test_lemma = train_test_split(
    df_lemma["text_lemma"],
    df_lemma["label"],
    test_size=0.2,
    random_state=42,
    stratify=df_lemma["label"]
)

print("Train size:", len(X_train_lemma))
print("Test size :", len(X_test_lemma))


# Load the FastText model already trained during data preparation
ft_lemma = fasttext.load_model(
    str(NEURAL_DATA_DIR / "fasttext.bin")
)

(   tokenizer_lemma,
    X_train_pad_lemma,
    X_test_pad_lemma,
    vocab_size_lemma,
    embedding_dim_lemma,
    embedding_matrix_lemma
) = prepare_sequences(
    X_train_lemma,
    X_test_lemma,
    ft_lemma,
    num_words=20000,
    max_len=100
)


print("X_train_pad:", X_train_pad_lemma.shape)
print("X_test_pad :", X_test_pad_lemma.shape)
print("Vocab size :", vocab_size_lemma)
print("Embedding matrix:", embedding_matrix_lemma.shape)


class_weights_lemma = create_class_weights(
    y_train_lemma
)

print("Class weights:", class_weights_lemma)

# Train Bidirectional GRU
rnn_lemma, history_rnn_lemma, rnn_pred_lemma, rnn_metrics_lemma = train_rnn(
    vocab_size_lemma,
    embedding_dim_lemma,
    embedding_matrix_lemma,
    X_train_pad_lemma,
    X_test_pad_lemma,
    y_train_lemma,
    y_test_lemma,
    class_weights_lemma,
    evaluate_model
)

rnn_result = pd.DataFrame([{
    "Preprocessing": "Lemmatization",
    "Model": "BiGRU",
    "Feature": "FastText 300d",
    "Accuracy": rnn_metrics_lemma["Accuracy"],
    "Precision": rnn_metrics_lemma["Precision"],
    "Recall": rnn_metrics_lemma["Recall"],
    "Macro F1": rnn_metrics_lemma["Macro F1"]
}])


print("\n********  BiGRU RESULT  ********\n")

print(
    rnn_result.to_string(index=False)
)

Train size: 50894
Test size : 12724
X_train_pad: (50894, 100)
X_test_pad : (12724, 100)
Vocab size : 20000
Embedding matrix: (20000, 300)
Class weights: {0: np.float64(1.3163149182702256), 1: np.float64(2.012654723771108), 2: np.float64(0.5735763149293933)}


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ (None, 100, 300)            │       6,000,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ (None, 256)                 │         330,240 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 64)                  │          16,448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 3)                   │             195 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,346,883 (24.21 MB)

 Trainable params: 6,346,883 (24.21 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
716/716 ━━━━━━━━━━━━━━━━━━━━ 190s 228ms/step - accuracy: 0.7015 - loss: 0.7520 - val_accuracy: 0.7566 - val_loss: 0.6076
Epoch 2/20
716/716 ━━━━━━━━━━━━━━━━━━━━ 158s 221ms/step - accuracy: 0.7656 - loss: 0.6442 - val_accuracy: 0.7672 - val_loss: 0.5779
Epoch 3/20
716/716 ━━━━━━━━━━━━━━━━━━━━ 150s 209ms/step - accuracy: 0.7869 - loss: 0.5993 - val_accuracy: 0.7862 - val_loss: 0.5358
Epoch 4/20
716/716 ━━━━━━━━━━━━━━━━━━━━ 157s 219ms/step - accuracy: 0.8062 - loss: 0.5525 - val_accuracy: 0.7383 - val_loss: 0.6288
Epoch 5/20
716/716 ━━━━━━━━━━━━━━━━━━━━ 161s 225ms/step - accuracy: 0.8231 - loss: 0.5115 - val_accuracy: 0.7688 - val_loss: 0.5967
Epoch 6/20
716/716 ━━━━━━━━━━━━━━━━━━━━ 162s 226ms/step - accuracy: 0.8393 - loss: 0.4690 - val_accuracy: 0.7658 - val_loss: 0.6260
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 3.
398/398 ━━━━━━━━━━━━━━━━━━━━ 15s 35ms/step

********  RNN RESULTS  ********
                 precision    recall  f1-score   

In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from src.utils import PROCESSED_DATA_DIR
from src.models.transformers import train_parsbert


df = pd.read_csv(
    PROCESSED_DATA_DIR / "parsbert_data.csv",
    encoding="utf-8-sig"
)

df_sample, _ = train_test_split(
    df,
    train_size=1000,
    random_state=42,
    stratify=df["label"]
)

print("Sample size:", len(df_sample))

print("\nClass distribution:")
print(
    df_sample["label"]
    .value_counts()
    .sort_index()
)

X_train, X_test, y_train, y_test = train_test_split(
    df_sample["text"],
    df_sample["label"],
    test_size=0.2,
    random_state=42,
    stratify=df_sample["label"]
)

print("\nTrain size:", len(X_train))
print("Test size :", len(X_test))

trainer_parsbert, parsbert_model, parsbert_tokenizer, parsbert_pred = train_parsbert(
    X_train,
    X_test,
    y_train,
    y_test
)

Sample size: 1000

Class distribution:
label
0    253
1    166
2    581
Name: count, dtype: int64

Train size: 800
Test size : 200
Using device: cpu

Train size     : 720
Validation size: 80
Test size      : 200
Class weights: [1.3186814  2.         0.57416266]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.051856,1.023984,0.800000,0.569072
2,0.524745,1.096321,0.800000,0.622879


Writing model shards:   0%|          | 0/1 [00:01<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


********  ParsBERT RESULTS  ********
                 precision    recall  f1-score   support

not_recommended       0.65      0.92      0.76        51
        no_idea       0.00      0.00      0.00        33
    recommended       0.85      0.94      0.89       116

       accuracy                           0.78       200
      macro avg       0.50      0.62      0.55       200
   weighted avg       0.66      0.78      0.71       200



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved.
